# ForenSight: Colab quick start

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

1. Setup (GPU check, Drive, install)
2. **Audio, no approval needed:** download ASVspoof 2019 LA, evaluate the pretrained AASIST, train the MelCNN baseline
3. **Video, after FaceForensics++ access is approved:** preprocess faces, train EfficientNet-B4

Results are written to `MyDrive/forensight/runs/`. Log each finished run in `docs/experiments.md`.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# The repo is private. Create a GitHub fine-grained token (Settings > Developer settings >
# Fine-grained tokens, repo ForenSight-FYP, Contents: read) and save it in Colab's
# Secrets panel (key icon on the left) as GITHUB_TOKEN with notebook access on.
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{TOKEN}@github.com/ForenSight-FYP/ForenSight-FYP.git"
!git clone -q $REPO_URL /content/forensight || (cd /content/forensight && git pull -q)
%cd /content/forensight
!pip install -q -e ".[dl,dev]"
!pytest -q

## 2. Audio module

### 2a. ASVspoof 2019 LA **evaluation** set (Hugging Face copy, ~4.4 GB, about 1 minute)
Edinburgh DataShare returns **403 Forbidden** to Colab, so the eval set comes from
`SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA` on Hugging Face: the official eval audio and the
official CM eval protocol, unmodified, redistributed under the same ODC-By licence.
The audio is stored in Parquet shards; this cell writes it back out as FLAC files.

In [ ]:
import os, glob, pyarrow.parquet as pq
from huggingface_hub import snapshot_download

OUT = "/content/asvspoof/LA19_eval"
snapshot_download("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA", repo_type="dataset", local_dir=OUT)
os.makedirs(f"{OUT}/flac", exist_ok=True)
for pf in sorted(glob.glob(f"{OUT}/data/test-*.parquet")):
    table = pq.read_table(pf, columns=["audio"])
    for row in table.column("audio").to_pylist():
        name = os.path.basename(row["path"])
        name = name if name.endswith(".flac") else name + ".flac"
        with open(f"{OUT}/flac/{name}", "wb") as f:
            f.write(row["bytes"])
PROTO = glob.glob(f"{OUT}/**/ASVspoof2019.LA.cm.eval.trl.txt", recursive=True)[0]
n = len(os.listdir(f"{OUT}/flac"))
print("clips:", n, "| labels:", PROTO)
assert n == 71237, "expected 71,237 eval clips"
LA19 = f"eval_sets={{la19_eval: {{protocol: {PROTO}, audio: {OUT}/flac}}}}"
%cd /content/forensight

### 2b. Pretrained AASIST on ASVspoof 2019 LA eval (pipeline sanity check)
The AASIST paper reports EER about 0.83%. **Our first run (2026-11-17): EER 0.82%, AUC 0.9993**
on all 71,237 clips, so data, labels, model wrapper and metrics are correct. Takes 10-30 min on a T4.

In [ ]:
!bash scripts/setup_aasist.sh
cmd = f'python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml "{LA19}"'
!{cmd}

### 2c. Train the MelCNN baseline: needs the **train/dev** sets from the full `LA.zip`
The Hugging Face copy has the eval set only. Get `LA.zip` once:
1. On your own computer, open https://datashare.ed.ac.uk/handle/10283/3336 in a browser and download **LA.zip** (several GB).
2. Upload it to the shared Google Drive as `MyDrive/forensight/datasets/asvspoof2019_LA.zip`.

The next cell uses that Drive copy (or tries DataShare directly, which usually fails from Colab).

In [ ]:
import os
URL = "https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip"
CACHE = "/content/drive/MyDrive/forensight/datasets/asvspoof2019_LA.zip"
LOCAL = "/content/LA.zip"
MIN_SIZE = 1024**3   # the real LA.zip is several GB; anything under 1 GB is a failed download/upload

def ok(path):
    return os.path.exists(path) and os.path.getsize(path) > MIN_SIZE

if ok(CACHE):
    print("Using LA.zip from Drive"); src = CACHE
else:
    if os.path.exists(CACHE):   # a broken earlier attempt
        os.remove(CACHE)
    !wget -c --progress=dot:giga -O {LOCAL} "{URL}" 2>&1 | tail -15
    if not ok(LOCAL):
        raise RuntimeError("No LA.zip: upload it to Drive as described above (DataShare blocks Colab).")
    !cp {LOCAL} {CACHE}
    src = LOCAL
!mkdir -p /content/asvspoof && unzip -q -n {src} -d /content/asvspoof
!ls /content/asvspoof/LA

In [ ]:
# smoke run first: 1 epoch, no extra test sets
!python -m forensight.training.train_audio --config configs/audio_melcnn.yaml train.epochs=1 eval_sets=null

In [ ]:
# full run (about 20 epochs), tested on the 2019 LA eval set from 2a
cmd = f'python -m forensight.training.train_audio --config configs/audio_melcnn.yaml "{LA19}"'
!{cmd}

### 2d. ASVspoof 2021 DF: cross-dataset test (generalisation, objective O3)
The full DF eval audio is 34.5 GB in four ~8.6 GB parts (Zenodo record 4835108), too big for a
free Drive. This downloads the official keys and **part00 only** straight to Colab's disk, and the
evaluation scores only the clips present (`only_existing: true`). Report it as
"ASVspoof 2021 DF eval, part00 subset". Add `01 02 03` to `PARTS` later if you have the disk space.

In [ ]:
PARTS = ["00"]
MD5 = {"00": "4f2cbae07cf3ede2a1dde3b8d2ee55ea", "01": "1578c89ab433c2b60b1dce93bdf8fbec",
       "02": "5497a35f0126e94a1d7a7d26db57b4f7", "03": "42b7512ba2943e98a32a53c9608cf83c"}
!mkdir -p /content/asvspoof/DF21
%cd /content/asvspoof/DF21
!wget -q -c https://www.asvspoof.org/asvspoof2021/DF-keys-full.tar.gz && tar -xzf DF-keys-full.tar.gz
for part in PARTS:
    f = f"ASVspoof2021_DF_eval_part{part}.tar.gz"
    !wget -q -c --show-progress -O {f} "https://zenodo.org/records/4835108/files/{f}?download=1"
    got = !md5sum {f}
    assert got[0].split()[0] == MD5[part], f"{f} is corrupted: delete it and re-run"
    !tar -xzf {f} && rm {f}
%cd /content/forensight
DF_KEYS = !find /content/asvspoof/DF21 -name trial_metadata.txt -path "*/CM/*"
DF_AUDIO = !find /content/asvspoof/DF21 -name "DF_E_*.flac" -print -quit
import os
DF_KEYS, DF_AUDIO = DF_KEYS[0], os.path.dirname(DF_AUDIO[0])
print(DF_KEYS, DF_AUDIO, len(os.listdir(DF_AUDIO)), "clips")

In [ ]:
# pretrained AASIST on the DF subset. Expect a much higher EER than on 2019 LA:
# that gap is exactly the generalisation problem in your problem statement.
OVR = f"eval_sets={{df21_eval: {{protocol: {DF_KEYS}, audio: {DF_AUDIO}, subset: eval, only_existing: true}}}}"
cmd = f'python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml "{OVR}"'
!{cmd}

In [ ]:
# your MelCNN on the same subset (after 2c has trained it)
RUN = "/content/drive/MyDrive/forensight/runs/audio_melcnn"
cmd = (f'python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml '
       f'model.name=melcnn model.checkpoint={RUN}/best.pt output_dir={RUN} "{OVR}"')
!{cmd}

## 3. Video module (needs FaceForensics++ access)
Put the downloaded `c23` videos and the official `splits/` folder in
`MyDrive/forensight/datasets/FaceForensics++/`.

In [ ]:
# faces -> local disk (one-off; takes a while). Use num_frames=8 for a quick first pass.
!python -m forensight.data.video_preprocess --config configs/video_effnet.yaml data.num_frames=8

In [ ]:
# smoke run on a small backbone, then the real run
!python -m forensight.training.train_video --config configs/video_effnet.yaml train.epochs=1 model.backbone=efficientnet_b0 data.image_size=224

In [ ]:
!python -m forensight.training.train_video --config configs/video_effnet.yaml